In [3]:
import torch
import torch.nn as nn
from tokenizers import Tokenizer
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
import numpy as np
import math
from attention import SmartDataCollator, TextClassificationDataset, EncoderModel, SkipConnection, evaluate_model, train_model, print_model_parameters

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [5]:
ds = load_dataset("stanfordnlp/sst2")

In [6]:
tokenizer = Tokenizer.from_file(
    f"results/bpe_{8000}.json"
)

Change the number to change which tokenizer is used.

In [7]:
embedding_dim = 512
hidden_dim = 128
output_dim = 512
batch_size = 128

In [8]:
collate_fn = SmartDataCollator(tokenizer)

dataset_train = TextClassificationDataset(list(ds["train"]["sentence"]), list(ds["train"]["label"]), tokenizer)
dataloader_train = DataLoader(dataset_train, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

dataset_val = TextClassificationDataset(list(ds["validation"]["sentence"]), list(ds["validation"]["label"]), tokenizer)
dataloader_val = DataLoader(dataset_val, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

For the test dataset, do the same as above

In [9]:
MLPmodel = nn.Sequential(
    EncoderModel(
        vocab_size=tokenizer.get_vocab_size(),
        embedding_dim=embedding_dim,
        hidden_dim=hidden_dim,
        output_dim=output_dim,
        pad_idx=tokenizer.token_to_id("[PAD]")
    ),
    SkipConnection(output_dim, output_dim // 2, nn.GELU()),
    nn.GELU(),
    SkipConnection(output_dim // 2, 1, nn.GELU()),
    nn.Sigmoid()
).to(device)
MLPoptimizer = torch.optim.AdamW(MLPmodel.parameters(), lr=1e-4)

In [10]:
print_model_parameters(MLPmodel)

Totale parameters: 13,278,722
Trainbare parameters: 13,278,722


In [11]:
Linearmodel = nn.Sequential(
    EncoderModel(
        vocab_size=tokenizer.get_vocab_size(),
        embedding_dim=embedding_dim,
        hidden_dim=hidden_dim,
        output_dim=output_dim,
        pad_idx=tokenizer.token_to_id("[PAD]")
    ),
    nn.Linear(output_dim, output_dim // 2),
    nn.GELU(),
    nn.Linear(output_dim // 2, 1),
    nn.Sigmoid()
).to(device)
Linearoptimizer = torch.optim.AdamW(Linearmodel.parameters(), lr=1e-4)

In [12]:
print_model_parameters(Linearmodel)

Totale parameters: 12,818,689
Trainbare parameters: 12,818,689


For different depths you can add more layers.  
You can change the parameters and learning rate.

In [39]:
criterion = nn.BCEWithLogitsLoss()

In [40]:
MLPmodel = train_model(MLPmodel, MLPoptimizer, dataloader_train, num_epochs=10, criterion=criterion, val_loader=dataloader_val, device=device)

Epoch 1/10 | Training Loss: 0.5930 | Validation Loss: 0.6100 | Validation Accuracy: 71.67%
Epoch 2/10 | Training Loss: 0.5493 | Validation Loss: 0.6017 | Validation Accuracy: 76.49%
Epoch 3/10 | Training Loss: 0.5363 | Validation Loss: 0.6043 | Validation Accuracy: 77.98%
Epoch 4/10 | Training Loss: 0.5261 | Validation Loss: 0.6070 | Validation Accuracy: 76.38%
Epoch 5/10 | Training Loss: 0.5208 | Validation Loss: 0.6004 | Validation Accuracy: 79.59%
Epoch 6/10 | Training Loss: 0.5171 | Validation Loss: 0.6000 | Validation Accuracy: 78.33%
Epoch 7/10 | Training Loss: 0.5129 | Validation Loss: 0.6086 | Validation Accuracy: 78.44%
Epoch 8/10 | Training Loss: 0.5103 | Validation Loss: 0.6051 | Validation Accuracy: 77.75%
Epoch 9/10 | Training Loss: 0.5087 | Validation Loss: 0.6062 | Validation Accuracy: 75.00%
Epoch 10/10 | Training Loss: 0.5087 | Validation Loss: 0.6046 | Validation Accuracy: 77.18%


In [41]:
Linearmodel = train_model(Linearmodel, Linearoptimizer, dataloader_train, num_epochs=10, criterion=criterion, val_loader=dataloader_val, device=device)

Epoch 1/10 | Training Loss: 0.5918 | Validation Loss: 0.6024 | Validation Accuracy: 78.10%
Epoch 2/10 | Training Loss: 0.5494 | Validation Loss: 0.6170 | Validation Accuracy: 76.83%
Epoch 3/10 | Training Loss: 0.5350 | Validation Loss: 0.5991 | Validation Accuracy: 79.01%
Epoch 4/10 | Training Loss: 0.5259 | Validation Loss: 0.6072 | Validation Accuracy: 77.87%
Epoch 5/10 | Training Loss: 0.5183 | Validation Loss: 0.5951 | Validation Accuracy: 78.33%
Epoch 6/10 | Training Loss: 0.5120 | Validation Loss: 0.6016 | Validation Accuracy: 78.78%
Epoch 7/10 | Training Loss: 0.5086 | Validation Loss: 0.6009 | Validation Accuracy: 78.44%
Epoch 8/10 | Training Loss: 0.5058 | Validation Loss: 0.6060 | Validation Accuracy: 78.33%
Epoch 9/10 | Training Loss: 0.5035 | Validation Loss: 0.5920 | Validation Accuracy: 79.93%
Epoch 10/10 | Training Loss: 0.5014 | Validation Loss: 0.6053 | Validation Accuracy: 77.64%
